# <center>CUDA PROGRAMMING LABORATORY</center>

---

## Laboratory Experiments

This notebook follows the same **beginner-friendly, step-by-step structure** as the supplied CUDA laboratory notebook.

### Experiments Covered

1. Parallel Sorting Algorithms
2. Parallel Merging
3. Communication between CPU and GPU
4. Mutual Exclusion between Two Threads
5. Peterson's Lock
6. Profiling CUDA Applications

Each experiment contains:
- Concept and objective
- Simple example
- Basic CUDA implementation
- Detailed CUDA implementation
- Compilation and execution commands
- Expected result
- Program outcome

---
<div align="center">

## EXPERIMENT 7
# PARALLEL SORTING ALGORITHMS

</div>

---

## Parallel Sorting Algorithms

**Parallel sorting** divides the sorting work among multiple GPU threads so that several comparisons can be performed simultaneously.

A simple algorithm suitable for learning CUDA is **Odd-Even Transposition Sort**.

For an array:

```text
[8, 3, 7, 4, 2, 6, 1, 5]
```

the algorithm repeatedly performs:

- **Even phase:** compare `(0,1), (2,3), (4,5), ...`
- **Odd phase:** compare `(1,2), (3,4), (5,6), ...`

After enough phases, the array becomes sorted.

### Why is it parallel?

During one phase, the pairs are independent:

```text
Even phase:

(0,1)   (2,3)   (4,5)   (6,7)
  ↓       ↓       ↓       ↓
compare  compare compare compare
```

Different GPU threads can process these pairs at the same time.

## Program 7.1: Basic Implementation

This program demonstrates Odd-Even Transposition Sort using CUDA. Each thread handles one comparison pair during each phase.

In [ ]:
%%writefile parallel_sort.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void oddEvenSort(int *A, int phase)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < N / 2)
    {
        int i;

        if (phase % 2 == 0)
            i = 2 * tid;
        else
            i = 2 * tid + 1;

        if (i + 1 < N && A[i] > A[i + 1])
        {
            int temp = A[i];
            A[i] = A[i + 1];
            A[i + 1] = temp;
        }
    }
}

int main()
{
    int h_A[N] = {8, 3, 7, 4, 2, 6, 1, 5};
    int *d_A;

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);

    for (int phase = 0; phase < N; phase++)
    {
        oddEvenSort<<<1, N / 2>>>(d_A, phase);
        cudaDeviceSynchronize();
    }

    cudaMemcpy(h_A, d_A, N * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Sorted Array:\n");
    for (int i = 0; i < N; i++)
        printf("%d ", h_A[i]);

    printf("\n");

    cudaFree(d_A);
    return 0;
}

In [ ]:
!nvcc parallel_sort.cu -o parallel_sort

In [ ]:
!./parallel_sort

## Program 7.2: Detailed Implementation

The detailed version displays the array after every phase so students can observe how the parallel sorting operation progresses.

In [ ]:
%%writefile parallel_sort_detail.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void oddEvenSort(int *A, int phase)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < N / 2)
    {
        int i = (phase % 2 == 0) ? 2 * tid : 2 * tid + 1;

        if (i + 1 < N && A[i] > A[i + 1])
        {
            int temp = A[i];
            A[i] = A[i + 1];
            A[i + 1] = temp;
        }
    }
}

void printArray(int *A)
{
    for (int i = 0; i < N; i++)
        printf("%d ", A[i]);
    printf("\n");
}

int main()
{
    int h_A[N] = {8, 3, 7, 4, 2, 6, 1, 5};
    int *d_A;

    printf("=========================================\n");
    printf("       PARALLEL ODD-EVEN SORT\n");
    printf("=========================================\n");

    printf("\nInitial Array:\n");
    printArray(h_A);

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);

    for (int phase = 0; phase < N; phase++)
    {
        printf("\nPhase %d (%s phase):\n",
               phase + 1,
               (phase % 2 == 0) ? "Even" : "Odd");

        oddEvenSort<<<1, N / 2>>>(d_A, phase);
        cudaDeviceSynchronize();

        cudaMemcpy(h_A, d_A, N * sizeof(int), cudaMemcpyDeviceToHost);
        printArray(h_A);
    }

    printf("\nFinal Sorted Array:\n");
    printArray(h_A);

    cudaFree(d_A);

    printf("\nProgram Executed Successfully.\n");
    return 0;
}

In [ ]:
!nvcc parallel_sort_detail.cu -o parallel_sort_detail

In [ ]:
!./parallel_sort_detail

## Program Outcome

After completing this experiment, you will be able to:

1. Understand parallel sorting.
2. Explain even and odd phases.
3. Map comparison pairs to CUDA threads.
4. Implement Odd-Even Transposition Sort using CUDA.
5. Observe synchronization between sorting phases.

---
<div align="center">

## EXPERIMENT 8
# PARALLEL MERGING

</div>

---

## Parallel Merging

**Merging** combines two already sorted arrays into one sorted array.

Example:

```text
A = [1, 4, 7, 10]
B = [2, 3, 8, 9]

Merged = [1, 2, 3, 4, 7, 8, 9, 10]
```

In a parallel merge, different GPU threads determine where elements belong in the final array.

A useful idea is:

```text
For each element:
    Find how many elements from the other array are smaller.
    This gives the element's final position.
```

This allows multiple output positions to be computed independently.

## Program 8.1: Basic Implementation

Each thread handles one element from the first array and one element from the second array. A binary-search-style count determines the final position.

In [ ]:
%%writefile parallel_merge.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 4
#define TOTAL 8

__device__ int lowerBound(int *A, int n, int value)
{
    int left = 0, right = n;

    while (left < right)
    {
        int mid = (left + right) / 2;

        if (A[mid] < value)
            left = mid + 1;
        else
            right = mid;
    }

    return left;
}

__global__ void parallelMerge(int *A, int *B, int *C)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < N)
    {
        int a = A[tid];
        int posA = tid + lowerBound(B, N, a);
        C[posA] = a;
    }

    if (tid < N)
    {
        int b = B[tid];
        int posB = tid + lowerBound(A, N, b);
        C[posB] = b;
    }
}

int main()
{
    int h_A[N] = {1, 4, 7, 10};
    int h_B[N] = {2, 3, 8, 9};
    int h_C[TOTAL] = {0};

    int *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));
    cudaMalloc((void**)&d_C, TOTAL * sizeof(int));

    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(int), cudaMemcpyHostToDevice);

    parallelMerge<<<1, N>>>(d_A, d_B, d_C);
    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, TOTAL * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Merged Array:\n");
    for (int i = 0; i < TOTAL; i++)
        printf("%d ", h_C[i]);

    printf("\n");

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

In [ ]:
!nvcc parallel_merge.cu -o parallel_merge

In [ ]:
!./parallel_merge

## Program 8.2: Detailed Implementation

In [ ]:
%%writefile parallel_merge_detail.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 4
#define TOTAL 8

__device__ int lowerBound(int *A, int n, int value)
{
    int left = 0, right = n;

    while (left < right)
    {
        int mid = (left + right) / 2;

        if (A[mid] < value)
            left = mid + 1;
        else
            right = mid;
    }

    return left;
}

__global__ void parallelMerge(int *A, int *B, int *C)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < N)
    {
        int a = A[tid];
        int smallerFromB = lowerBound(B, N, a);
        int finalPosA = tid + smallerFromB;

        C[finalPosA] = a;

        printf("Thread %d: A[%d]=%d -> C[%d]\n",
               tid, tid, a, finalPosA);
    }

    if (tid < N)
    {
        int b = B[tid];
        int smallerFromA = lowerBound(A, N, b);
        int finalPosB = tid + smallerFromA;

        C[finalPosB] = b;

        printf("Thread %d: B[%d]=%d -> C[%d]\n",
               tid, tid, b, finalPosB);
    }
}

int main()
{
    int h_A[N] = {1, 4, 7, 10};
    int h_B[N] = {2, 3, 8, 9};
    int h_C[TOTAL] = {0};

    int *d_A, *d_B, *d_C;

    printf("=========================================\n");
    printf("             PARALLEL MERGING\n");
    printf("=========================================\n");

    printf("\nArray A: ");
    for (int i = 0; i < N; i++)
        printf("%d ", h_A[i]);

    printf("\nArray B: ");
    for (int i = 0; i < N; i++)
        printf("%d ", h_B[i]);

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));
    cudaMalloc((void**)&d_C, TOTAL * sizeof(int));

    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(int), cudaMemcpyHostToDevice);

    printf("\n\nLaunching parallel merge kernel...\n");

    parallelMerge<<<1, N>>>(d_A, d_B, d_C);
    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, TOTAL * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nFinal Merged Array:\n");
    for (int i = 0; i < TOTAL; i++)
        printf("C[%d] = %d\n", i, h_C[i]);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    printf("\nProgram Executed Successfully.\n");
    return 0;
}

In [ ]:
!nvcc parallel_merge_detail.cu -o parallel_merge_detail

In [ ]:
!./parallel_merge_detail

## Program Outcome

After completing this experiment, you will be able to:

1. Explain merging of sorted arrays.
2. Understand how an element can independently determine its final position.
3. Use binary search inside a CUDA kernel.
4. Implement a basic parallel merge.

---
<div align="center">

## EXPERIMENT 9
# COMMUNICATION BETWEEN CPU AND GPU

</div>

---

## Communication between CPU and GPU

CUDA programs normally use two memory spaces:

- **Host memory** → CPU memory
- **Device memory** → GPU memory

Communication happens through explicit data transfers.

### Main operations

```text
CPU memory
    |
    | cudaMemcpyHostToDevice
    v
GPU memory
    |
    | Kernel executes
    v
GPU memory
    |
    | cudaMemcpyDeviceToHost
    v
CPU memory
```

The two most important directions are:

```text
CPU → GPU : cudaMemcpyHostToDevice

GPU → CPU : cudaMemcpyDeviceToHost
```

## Program 9.1: Basic CPU-GPU Communication

In [ ]:
%%writefile cpu_gpu_communication.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void squareVector(int *A, int *B)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N)
        B[idx] = A[idx] * A[idx];
}

int main()
{
    int h_A[N] = {1,2,3,4,5,6,7,8};
    int h_B[N];

    int *d_A, *d_B;

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));

    // CPU -> GPU
    cudaMemcpy(d_A, h_A, N * sizeof(int),
               cudaMemcpyHostToDevice);

    squareVector<<<1, N>>>(d_A, d_B);
    cudaDeviceSynchronize();

    // GPU -> CPU
    cudaMemcpy(h_B, d_B, N * sizeof(int),
               cudaMemcpyDeviceToHost);

    printf("Result received by CPU:\n");

    for (int i = 0; i < N; i++)
        printf("%d ", h_B[i]);

    printf("\n");

    cudaFree(d_A);
    cudaFree(d_B);

    return 0;
}

In [ ]:
!nvcc cpu_gpu_communication.cu -o cpu_gpu_communication

In [ ]:
!./cpu_gpu_communication

## Program 9.2: Detailed CPU-GPU Communication

In [ ]:
%%writefile cpu_gpu_communication_detail.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void squareVector(int *A, int *B)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N)
    {
        B[idx] = A[idx] * A[idx];

        printf("GPU Thread %d: %d x %d = %d\n",
               idx, A[idx], A[idx], B[idx]);
    }
}

int main()
{
    int h_A[N] = {1,2,3,4,5,6,7,8};
    int h_B[N];

    int *d_A, *d_B;

    printf("=========================================\n");
    printf("       CPU-GPU COMMUNICATION\n");
    printf("=========================================\n");

    printf("\nCPU: Input data\n");
    for (int i = 0; i < N; i++)
        printf("%d ", h_A[i]);

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));

    printf("\n\nCPU -> GPU: Copying input data...\n");

    cudaMemcpy(d_A, h_A, N * sizeof(int),
               cudaMemcpyHostToDevice);

    printf("Launching GPU kernel...\n");

    squareVector<<<1, N>>>(d_A, d_B);
    cudaDeviceSynchronize();

    printf("\nGPU -> CPU: Copying results...\n");

    cudaMemcpy(h_B, d_B, N * sizeof(int),
               cudaMemcpyDeviceToHost);

    printf("\nCPU received:\n");
    for (int i = 0; i < N; i++)
        printf("B[%d] = %d\n", i, h_B[i]);

    cudaFree(d_A);
    cudaFree(d_B);

    printf("\nCommunication Completed Successfully.\n");

    return 0;
}

In [ ]:
!nvcc cpu_gpu_communication_detail.cu -o cpu_gpu_communication_detail

In [ ]:
!./cpu_gpu_communication_detail

## Program Outcome

After completing this experiment, you will be able to:

1. Differentiate host and device memory.
2. Explain CPU-to-GPU communication.
3. Explain GPU-to-CPU communication.
4. Use `cudaMemcpy()` correctly.
5. Understand where kernel execution occurs.

---
<div align="center">

## EXPERIMENT 10
# MUTUAL EXCLUSION BETWEEN TWO THREADS

</div>

---

## Mutual Exclusion

**Mutual exclusion** means that only one thread is allowed to enter a **critical section** at a time.

Consider two threads:

```text
Thread 0 ----              >---- Critical Section
Thread 1 ----/
```

If both threads modify the same shared variable at the same time, a **race condition** can occur.

A lock provides:

```text
Thread 0 → acquire lock → critical section → release lock

Thread 1 → waits --------------------------→ enters later
```

This experiment uses two CUDA threads and a simple spin lock based on `atomicExch()`.

## Program 10.1: Basic Mutual Exclusion using a CUDA Spin Lock

In [ ]:
%%writefile mutual_exclusion.cu

#include <stdio.h>
#include <cuda_runtime.h>

__device__ int lock = 0;
__device__ int counter = 0;

__device__ void acquireLock()
{
    while (atomicExch(&lock, 1) == 1)
    {
        // Wait until lock becomes free
    }
}

__device__ void releaseLock()
{
    atomicExch(&lock, 0);
}

__global__ void criticalSection()
{
    int tid = threadIdx.x;

    if (tid < 2)
    {
        acquireLock();

        int old = counter;
        counter = old + 1;

        printf("Thread %d entered critical section. Counter = %d\n",
               tid, counter);

        releaseLock();
    }
}

int main()
{
    criticalSection<<<1, 2>>>();
    cudaDeviceSynchronize();

    int h_counter;
    cudaMemcpyFromSymbol(&h_counter, counter,
                         sizeof(int), 0,
                         cudaMemcpyDeviceToHost);

    printf("Final Counter = %d\n", h_counter);

    return 0;
}

In [ ]:
!nvcc mutual_exclusion.cu -o mutual_exclusion

In [ ]:
!./mutual_exclusion

## Program 10.2: Detailed Mutual Exclusion

The following version clearly displays when each thread acquires and releases the lock.

In [ ]:
%%writefile mutual_exclusion_detail.cu

#include <stdio.h>
#include <cuda_runtime.h>

__device__ int lock = 0;
__device__ int counter = 0;

__device__ void acquireLock()
{
    while (atomicExch(&lock, 1) == 1)
    {
    }
}

__device__ void releaseLock()
{
    atomicExch(&lock, 0);
}

__global__ void criticalSection()
{
    int tid = threadIdx.x;

    if (tid < 2)
    {
        printf("Thread %d: waiting for lock...\n", tid);

        acquireLock();

        printf("Thread %d: LOCK ACQUIRED\n", tid);

        int old = counter;
        counter = old + 1;

        printf("Thread %d: critical section, counter = %d\n",
               tid, counter);

        releaseLock();

        printf("Thread %d: LOCK RELEASED\n", tid);
    }
}

int main()
{
    printf("=========================================\n");
    printf("        MUTUAL EXCLUSION - CUDA\n");
    printf("=========================================\n");

    criticalSection<<<1, 2>>>();
    cudaDeviceSynchronize();

    int h_counter;

    cudaMemcpyFromSymbol(&h_counter, counter,
                         sizeof(int), 0,
                         cudaMemcpyDeviceToHost);

    printf("\nFinal Counter = %d\n", h_counter);
    printf("Only one thread enters the critical section at a time.\n");

    return 0;
}

In [ ]:
!nvcc mutual_exclusion_detail.cu -o mutual_exclusion_detail

In [ ]:
!./mutual_exclusion_detail

## Program Outcome

After completing this experiment, you will be able to:

1. Define mutual exclusion.
2. Explain a critical section.
3. Identify race conditions.
4. Use an atomic operation to implement a simple CUDA spin lock.
5. Understand why synchronization is required for shared data.

---
<div align="center">

## EXPERIMENT 11
# PETERSON'S LOCK

</div>

---

## Peterson's Lock

**Peterson's algorithm** is a classical software solution for mutual exclusion between **two threads**.

It uses two shared variables:

```text
flag[0], flag[1]
turn
```

### Meaning

```text
flag[i] = 1
```

means:

> Thread i wants to enter the critical section.

The `turn` variable indicates which thread should get priority when both threads want to enter.

### Basic idea

For Thread 0:

```text
flag[0] = true
turn = 1

while(flag[1] && turn == 1)
    wait
```

For Thread 1:

```text
flag[1] = true
turn = 0

while(flag[0] && turn == 0)
    wait
```

Only one of the two threads can pass the waiting condition.

> **CUDA note:** GPU memory ordering is different from the simple textbook CPU model. The implementation below uses CUDA atomic exchange operations and a memory fence to make the educational Peterson-style lock safer on CUDA. It is intended for exactly two participating threads.

## Program 11.1: Peterson's Lock with Two CUDA Threads

In [ ]:
%%writefile peterson_lock.cu

#include <stdio.h>
#include <cuda_runtime.h>

__device__ volatile int flag[2] = {0, 0};
__device__ volatile int turn = 0;
__device__ int sharedValue = 0;

__device__ void peterson_lock(int tid)
{
    int other = 1 - tid;

    atomicExch((int*)&flag[tid], 1);
    __threadfence();

    atomicExch((int*)&turn, other);
    __threadfence();

    while (flag[other] && turn == other)
    {
    }
}

__device__ void peterson_unlock(int tid)
{
    __threadfence();
    atomicExch((int*)&flag[tid], 0);
}

__global__ void testPeterson()
{
    int tid = threadIdx.x;

    if (tid < 2)
    {
        peterson_lock(tid);

        int old = sharedValue;
        sharedValue = old + 1;

        printf("Thread %d entered critical section. sharedValue = %d\n",
               tid, sharedValue);

        peterson_unlock(tid);
    }
}

int main()
{
    testPeterson<<<1, 2>>>();
    cudaDeviceSynchronize();

    int h_value;

    cudaMemcpyFromSymbol(&h_value, sharedValue,
                         sizeof(int), 0,
                         cudaMemcpyDeviceToHost);

    printf("Final sharedValue = %d\n", h_value);

    return 0;
}

In [ ]:
!nvcc peterson_lock.cu -o peterson_lock

In [ ]:
!./peterson_lock

## Program 11.2: Detailed Peterson's Lock

In [ ]:
%%writefile peterson_lock_detail.cu

#include <stdio.h>
#include <cuda_runtime.h>

__device__ volatile int flag[2] = {0, 0};
__device__ volatile int turn = 0;
__device__ int sharedValue = 0;

__device__ void peterson_lock(int tid)
{
    int other = 1 - tid;

    printf("Thread %d: wants to enter\n", tid);

    atomicExch((int*)&flag[tid], 1);
    __threadfence();

    atomicExch((int*)&turn, other);
    __threadfence();

    while (flag[other] && turn == other)
    {
    }

    printf("Thread %d: LOCK ACQUIRED\n", tid);
}

__device__ void peterson_unlock(int tid)
{
    __threadfence();
    atomicExch((int*)&flag[tid], 0);

    printf("Thread %d: LOCK RELEASED\n", tid);
}

__global__ void testPeterson()
{
    int tid = threadIdx.x;

    if (tid < 2)
    {
        peterson_lock(tid);

        int old = sharedValue;
        sharedValue = old + 1;

        printf("Thread %d: critical section -> sharedValue = %d\n",
               tid, sharedValue);

        peterson_unlock(tid);
    }
}

int main()
{
    printf("=========================================\n");
    printf("          PETERSON'S LOCK - CUDA\n");
    printf("=========================================\n");

    testPeterson<<<1, 2>>>();
    cudaDeviceSynchronize();

    int h_value;

    cudaMemcpyFromSymbol(&h_value, sharedValue,
                         sizeof(int), 0,
                         cudaMemcpyDeviceToHost);

    printf("\nFinal sharedValue = %d\n", h_value);
    printf("Two threads used Peterson-style mutual exclusion.\n");

    return 0;
}

In [ ]:
!nvcc peterson_lock_detail.cu -o peterson_lock_detail

In [ ]:
!./peterson_lock_detail

## Program Outcome

After completing this experiment, you will be able to:

1. Explain Peterson's mutual exclusion algorithm.
2. Explain the roles of `flag[]` and `turn`.
3. Understand how two threads coordinate entry to a critical section.
4. Relate the classical Peterson algorithm to CUDA memory ordering.
5. Distinguish a software lock from a hardware atomic operation.

---
<div align="center">

## EXPERIMENT 12
# PROFILING CUDA APPLICATIONS

</div>

---

## Profiling CUDA Applications

**Profiling** means measuring a CUDA program to understand where execution time is being spent and how efficiently the GPU is being used.

Important measurements include:

- Kernel execution time
- Memory transfer time
- Number of kernel launches
- GPU utilization
- Memory usage
- Synchronization overhead

A simple and portable CUDA method is to use **CUDA Events** for accurate GPU timing.

```text
CPU
 |
 | Start Event
 v
GPU Kernel
 |
 | Stop Event
 v
Elapsed GPU Time
```

External tools such as **NVIDIA Nsight Systems** can also be used when they are available in the CUDA environment.

## Program 12.1: Basic CUDA Profiling using CUDA Events

In [ ]:
%%writefile profile_cuda.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 1000000
#define THREADS 256

__global__ void vectorAdd(float *A, float *B, float *C)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N)
        C[idx] = A[idx] + B[idx];
}

int main()
{
    size_t bytes = N * sizeof(float);

    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);

    for (int i = 0; i < N; i++)
    {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, bytes);
    cudaMalloc((void**)&d_B, bytes);
    cudaMalloc((void**)&d_C, bytes);

    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS - 1) / THREADS;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    vectorAdd<<<blocks, THREADS>>>(d_A, d_B, d_C);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    printf("Kernel Execution Time = %.3f ms\n", milliseconds);

    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    printf("Sample Result: C[0] = %.1f\n", h_C[0]);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

In [ ]:
!nvcc -O2 profile_cuda.cu -o profile_cuda

In [ ]:
!./profile_cuda

## Program 12.2: Detailed Profiling

This version measures both the CPU-to-GPU transfer and the kernel execution using CUDA Events.

In [ ]:
%%writefile profile_cuda_detail.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 1000000
#define THREADS 256

__global__ void vectorAdd(float *A, float *B, float *C)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N)
        C[idx] = A[idx] + B[idx];
}

int main()
{
    size_t bytes = N * sizeof(float);

    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);

    for (int i = 0; i < N; i++)
    {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, bytes);
    cudaMalloc((void**)&d_B, bytes);
    cudaMalloc((void**)&d_C, bytes);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    printf("=========================================\n");
    printf("          CUDA APPLICATION PROFILING\n");
    printf("=========================================\n");

    // Measure Host -> Device transfer
    cudaEventRecord(start);

    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float transferTime = 0;
    cudaEventElapsedTime(&transferTime, start, stop);

    printf("\nCPU -> GPU Transfer Time = %.3f ms\n",
           transferTime);

    int blocks = (N + THREADS - 1) / THREADS;

    // Measure kernel
    cudaEventRecord(start);

    vectorAdd<<<blocks, THREADS>>>(d_A, d_B, d_C);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float kernelTime = 0;
    cudaEventElapsedTime(&kernelTime, start, stop);

    printf("GPU Kernel Time           = %.3f ms\n",
           kernelTime);

    // Measure Device -> Host transfer
    cudaEventRecord(start);

    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float copyBackTime = 0;
    cudaEventElapsedTime(&copyBackTime, start, stop);

    printf("GPU -> CPU Transfer Time = %.3f ms\n",
           copyBackTime);

    printf("\nSample Result: C[0] = %.1f\n", h_C[0]);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    printf("\nProfiling Completed Successfully.\n");

    return 0;
}

In [ ]:
!nvcc -O2 profile_cuda_detail.cu -o profile_cuda_detail

In [ ]:
!./profile_cuda_detail

## Optional External Profiling

If the CUDA environment provides NVIDIA Nsight Systems, an application can be profiled with a command such as:

```bash
nsys profile -o cuda_profile ./profile_cuda
```

The generated report can be opened with NVIDIA Nsight Systems on a compatible system.

If `nsys` is not installed in the notebook environment, use the CUDA Event timing program above instead.

## Program Outcome

After completing this experiment, you will be able to:

1. Define CUDA profiling.
2. Measure GPU kernel execution time.
3. Measure CPU-GPU transfer time.
4. Use CUDA Events for timing.
5. Identify the importance of memory-transfer and kernel-execution costs.
6. Understand the purpose of external CUDA profiling tools.

---
# LABORATORY QUESTIONS / VIVA QUESTIONS

1. What is parallel sorting?
2. Why is Odd-Even Transposition Sort suitable for demonstrating CUDA parallelism?
3. What is parallel merging?
4. How can an element determine its position in a merged sorted array?
5. What is the difference between Host memory and Device memory?
6. What is the purpose of `cudaMemcpyHostToDevice`?
7. What is the purpose of `cudaMemcpyDeviceToHost`?
8. What is mutual exclusion?
9. What is a critical section?
10. What is a race condition?
11. Why are atomic operations useful for implementing locks?
12. What is Peterson's algorithm?
13. What are the roles of `flag[]` and `turn` in Peterson's algorithm?
14. Why does Peterson's algorithm involve exactly two participating threads?
15. What is CUDA profiling?
16. Why are CUDA Events useful for measuring kernel execution time?
17. What is the difference between kernel execution time and CPU-GPU data-transfer time?
18. What information can external CUDA profilers provide?